# Extracting Categories  
A preprocessign step defining training data

In [2]:
import duckdb as ddb
import json
import pandas as pd
from pathlib import Path

from classifier.preparation import DataProcessor

In [3]:
# load configs
with open(Path.cwd().parent/"data"/"config.json", "r", encoding="utf-8") as f:
    config = json.load(f)
db_parquet = Path(config["original_db"])  # extract path to db

In [4]:
# load data
res_df = pd.DataFrame(ddb.sql(
    f"SELECT code, product_name, categories_tags FROM read_parquet('{db_parquet.resolve().as_posix()}')"
).df())

res_df.head()

,code,product_name,categories_tags
0,0000101209159,"[{'lang': 'main', 'text': 'Véritable pâte à ta...","[en:breakfasts, en:spreads, en:sweet-spreads, ..."
1,0000105000011,"[{'lang': 'main', 'text': 'Chamomile Herbal Te...",[en:null]
2,0000105000042,"[{'lang': 'main', 'text': 'Lagg's, herbal tea,...","[en:plant-based-foods-and-beverages, en:bevera..."
3,0000105000059,"[{'lang': 'main', 'text': 'Linden Flowers Tea'...","[en:beverages-and-beverages-preparations, en:p..."
4,0000105000073,"[{'lang': 'main', 'text': 'Herbal Tea, Hibiscu...",[]


In [5]:

min_occurances_of_tags = config["minimum_tag_ocurances"]
min_occurances_of_tags

10

In [6]:
all_tags = []

for tags in res_df["categories_tags"].dropna():
    for tag in tags:
        if tag:
            all_tags.append(tag)

tag_counts = pd.Series(all_tags, dtype="string").value_counts()

allowed_tags = tag_counts[
    (tag_counts >= config["minimum_tag_ocurances"]) # filter for occurances if not german 
    | tag_counts.index.str.startswith("de:")        # keep all german tags
]

# normalize tags
normalized_tags = (
    allowed_tags.index
    .str.lower()                                    # ensure lowercase
    .str.replace(" - ", "", regex=False)            # remove "-" variations
    .str.replace("-", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace("|", "", regex=False)
    .str.replace("ä","a", regex=False)
    .str.replace("ö","o", regex=False)
    .str.replace("ü","u", regex=False)

)

unique_labels = sorted(set(normalized_tags))
len(unique_labels)

18156

In [7]:
with open(config["vocab_json"], "w", encoding="utf-8") as f:
    json.dump(unique_labels, f, ensure_ascii=False)

In [8]:
pd.Series(normalized_tags).str[:2].value_counts()

de    9459
en    7737
fr    1417
nl     446
es     312
it     221
pt      50
ru      36
ja      36
sk      35
ro      19
xx      18
pl      17
el      15
bg      12
uk      12
th       8
da       5
cs       4
ca       2
hr       2
sv       2
sl       2
af       1
ar       1
gr       1
fi       1
az       1
lv       1
zh       1
lt       1
Name: count, dtype: int64[pyarrow]

In [9]:
# create file to let LLM map database entries
export_path = Path(config["let_llm_map"])
has_tags = res_df["categories_tags"].str.len().fillna(0) > 0    # only thos WITH tags -> otherwise could no tags lead to not-classification to rest-class (which makes no sense, since info is to be extracted from PRESENT labels)
res_df.loc[has_tags, ["product_name", "code", "categories_tags"]].to_csv(
    export_path, index=False, encoding="utf-8"
)
export_path


WindowsPath('C:/Users/Emil/Documents/Repositories/FoodCLS/data/let_llm_map.csv')